# Обучение модели ResNet18 для классификации AI vs Human Generated Images

Этот notebook содержит код для обучения модели ResNet18 на датасете Train_1 и оценки качества на Test_1.

## 1. Импорт библиотек

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 2. Подготовка данных

In [3]:
class ImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path = self.root_dir / self.data.iloc[idx]['file_name']
        image = Image.open(img_path).convert('RGB')
        label = self.data.iloc[idx]['label']
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [19]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset-hw/Train_1/train.csv',
    root_dir='ai-vs-human-generated-dataset-hw/Train_1',
    transform=train_transform
)

test_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset-hw/Test_1/test.csv',
    root_dir='ai-vs-human-generated-dataset-hw/Test_1',
    transform=test_transform
)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')

Train dataset size: 9993
Test dataset size: 3997


## 3. Создание модели ResNet18

In [24]:
model = models.resnet18()
model.load_state_dict(torch.load("./resnet18-f37072fd.pth"))
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

print(f'Model architecture:')
print(model)

Model architecture:
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=Tru

## 4. Определение функции потерь и оптимизатора

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

## 5. Функция обучения

In [7]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for images, labels in tqdm(dataloader, desc='Training'):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    # подсчет epoch_loss, epoch_acc, epoch_f1
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds)

    return epoch_loss, epoch_acc, epoch_f1

## 6. Функция валидации

In [8]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # подсчет epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds)
    epoch_precision = precision_score(all_labels, all_preds)
    epoch_recall = recall_score(all_labels, all_preds)
    
    return epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall

## 7. Обучение модели

В фоне запущен `tensorboard --logdir=my_logs`

In [10]:
num_epochs = 10
train_losses = []
train_accs = []
train_f1s = []

# Добавьте логгирование метрик для tensorboard
writer = SummaryWriter('my_logs/resnet18')

model.train()
for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 50)
    
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)
    scheduler.step()
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    train_f1s.append(train_f1)

    writer.add_scalar("loss/train", train_loss, epoch)
    writer.add_scalar("acc/train", train_acc, epoch)
    writer.add_scalar("f1/train", train_f1, epoch)
    
    print(f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}')

print('\nTraining completed!')


Epoch 1/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [00:49<00:00,  6.37it/s]


Train Loss: 11.8907, Acc: 0.8432, F1: 0.8432

Epoch 2/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [00:48<00:00,  6.43it/s]


Train Loss: 8.7606, Acc: 0.8863, F1: 0.8859

Epoch 3/10
--------------------------------------------------


...

Train Loss: 3.2063, Acc: 0.9620, F1: 0.9620

Epoch 9/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [00:48<00:00,  6.44it/s]


Train Loss: 3.3218, Acc: 0.9585, F1: 0.9584

Epoch 10/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [00:52<00:00,  5.94it/s]

Train Loss: 2.6867, Acc: 0.9707, F1: 0.9707

Training completed!


## 8. Оценка модели на тестовом датасете

In [22]:
model.eval()
test_loss, test_acc, test_f1, test_precision, test_recall = validate(model, test_loader, criterion, device)
print(
    f"test_loss: {test_loss}",
    f"test_acc: {test_acc}",
    f"test_f1: {test_f1}",
    f"test_precision: {test_precision}",
    f"test_recall: {test_recall}",
    sep="\n"
)

Validation: 100%|██████████| 125/125 [00:07<00:00, 16.93it/s]

test_loss: 3.1148130714893343
test_acc: 0.9687265449086815
test_f1: 0.9684263702955291
test_precision: 0.9790602655771196
test_recall: 0.9580209895052474


## 9. Дообучите модель на втором датасете и постройте DVC пайплайн

См. пункт 4 из README.md

## 10. Напишите вывод о полученных результатах

Использование tensorboard позволило получить наглядные логи обучения с минимальными усилиями, что говорит о полезности инструмента

Использование S3 мне никакого выхлопа не дало, но потенциально полезно при работе в команде (удобно делиться моделями)

Построение dvc-пайплайна дало возможность быстро воспроизводить обучение моделей. 
Особенно полезной мне показалась идея зависимостей целей в dvc, которая позволяет не перезапускать полностью пайплайн, если какие-то его стадии уже были выполнены (напоминает инкрементальную сборку в make).

Что касается моделей, результаты обучения+дообучения следующие:
```
                                            test_acc       test_f1     test_loss  test_precision   test_recall
--------------------------------------------------------------------------------------------------------------
metrics/metrics_of_trained.json              0.97223       0.97223       2.52490         0.97345       0.97101
metrics/metrics_of_post_trained.json         0.96750       0.96682       2.96814         0.98237       0.95176
```
После дообучения точность, f1 и recall немного снизились, precision вырос. Считаю эти изменения не значительными из-за небольшого числа эпох в экспериментах.  